# 🧠 Notebook 04 — Visualization

**Goal:** Turn the cleaned NeuroVault data into clear, professional charts that tell a story.

**What you will learn:**
- How to create bar charts, line plots, pie charts and heatmaps
- How to style charts professionally
- How to add titles, labels and annotations
- How to save charts as PNG files

**Input:** `data/processed/collections_clean.csv` + `data/processed/images_clean.csv`

---
## 1. Setup — Mount Drive & Import libraries

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print('Google Drive mounted ✅')

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import os
import warnings
warnings.filterwarnings('ignore')

# Global style
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.family'] = 'DejaVu Sans'
sns.set_theme(style='darkgrid', palette='muted')

# Output folder for charts
CHARTS_PATH = '/content/drive/MyDrive/neurovault/charts'
os.makedirs(CHARTS_PATH, exist_ok=True)

print('Libraries imported ✅')
print(f'Charts will be saved to: {CHARTS_PATH}')

---
## 2. Load clean data

In [ ]:
BASE = '/content/drive/MyDrive/neurovault/processed'

df_c = pd.read_csv(f'{BASE}/collections_clean.csv', low_memory=False)
df_i = pd.read_csv(f'{BASE}/images_clean.csv', low_memory=False)

print(f'Collections: {df_c.shape[0]:,} rows × {df_c.shape[1]} columns')
print(f'Images:      {df_i.shape[0]:,} rows × {df_i.shape[1]} columns')

---
## 3. Chart 1 — Studies published per year (Bar chart)

Shows how neuroscience research has grown over time.

In [ ]:
yearly = df_c.groupby('year')['id'].count().reset_index()
yearly.columns = ['year', 'total']
yearly = yearly[yearly['year'] >= 2013]

fig, ax = plt.subplots(figsize=(14, 6))

bars = ax.bar(yearly['year'], yearly['total'],
              color=sns.color_palette('Blues_d', len(yearly)),
              edgecolor='white', linewidth=0.5)

# Highlight 2020 and 2023 peaks
for bar, year in zip(bars, yearly['year']):
    if year in [2020, 2023, 2024]:
        bar.set_color('#E84393')

# Add value labels on top of bars
for bar in bars:
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., height + 20,
            f'{int(height):,}', ha='center', va='bottom', fontsize=9)

ax.set_title('Brain Imaging Studies Published per Year on NeuroVault', fontsize=16, fontweight='bold', pad=20)
ax.set_xlabel('Year', fontsize=12)
ax.set_ylabel('Number of Studies', fontsize=12)
ax.xaxis.set_major_locator(mticker.MultipleLocator(1))
ax.annotate('COVID-19 boom', xy=(2020, 2560), xytext=(2018.5, 2800),
            arrowprops=dict(arrowstyle='->', color='gray'), fontsize=10, color='gray')
ax.annotate('AI in neuroscience', xy=(2023, 2197), xytext=(2021.5, 2450),
            arrowprops=dict(arrowstyle='->', color='gray'), fontsize=10, color='gray')

plt.tight_layout()
plt.savefig(f'{CHARTS_PATH}/01_studies_per_year.png', bbox_inches='tight')
plt.show()
print('Chart saved ✅')

---
## 4. Chart 2 — Growth rate year over year (Line chart)

In [ ]:
yearly['growth'] = yearly['total'].pct_change() * 100
growth = yearly.dropna()

fig, ax = plt.subplots(figsize=(14, 6))

colors = ['#E84393' if g > 0 else '#6C757D' for g in growth['growth']]
ax.bar(growth['year'], growth['growth'], color=colors, edgecolor='white', alpha=0.8)
ax.plot(growth['year'], growth['growth'], color='white', linewidth=2, marker='o', markersize=5)
ax.axhline(y=0, color='white', linewidth=0.8, linestyle='--', alpha=0.5)

ax.set_title('Year-over-Year Growth Rate of Brain Imaging Studies (%)', fontsize=16, fontweight='bold', pad=20)
ax.set_xlabel('Year', fontsize=12)
ax.set_ylabel('Growth (%)', fontsize=12)
ax.xaxis.set_major_locator(mticker.MultipleLocator(1))

for x, y in zip(growth['year'], growth['growth']):
    ax.text(x, y + (15 if y >= 0 else -25), f'{y:.0f}%',
            ha='center', fontsize=8, color='white')

plt.tight_layout()
plt.savefig(f'{CHARTS_PATH}/02_growth_rate.png', bbox_inches='tight')
plt.show()
print('Chart saved ✅')

---
## 5. Chart 3 — Image modality breakdown (Pie + Bar)

In [ ]:
modality = df_i['modality'].value_counts().reset_index()
modality.columns = ['modality', 'count']

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 7))

# Pie chart
colors_pie = sns.color_palette('Set2', len(modality))
wedges, texts, autotexts = ax1.pie(
    modality['count'],
    labels=modality['modality'],
    autopct='%1.1f%%',
    colors=colors_pie,
    startangle=90,
    pctdistance=0.85
)
for text in autotexts:
    text.set_fontsize(9)
ax1.set_title('Brain Image Modality Distribution', fontsize=14, fontweight='bold')

# Bar chart
bars = ax2.barh(modality['modality'], modality['count'],
                color=colors_pie, edgecolor='white')
for bar in bars:
    width = bar.get_width()
    ax2.text(width + 10, bar.get_y() + bar.get_height()/2,
             f'{int(width):,}', va='center', fontsize=10)
ax2.set_title('Count by Modality', fontsize=14, fontweight='bold')
ax2.set_xlabel('Number of Images')
ax2.invert_yaxis()

plt.suptitle('NeuroVault — Brain Imaging Modalities', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(f'{CHARTS_PATH}/03_modality_breakdown.png', bbox_inches='tight')
plt.show()
print('Chart saved ✅')

---
## 6. Chart 4 — DOI coverage over the years

In [ ]:
doi_year = df_c.groupby('year').agg(
    total=('id', 'count'),
    with_doi=('has_doi', 'sum')
).reset_index()
doi_year['pct_doi'] = (doi_year['with_doi'] / doi_year['total'] * 100).round(1)
doi_year = doi_year[doi_year['year'] >= 2013]

fig, ax1 = plt.subplots(figsize=(14, 6))

ax2 = ax1.twinx()
ax1.bar(doi_year['year'], doi_year['total'], color='#4C8BE8', alpha=0.6, label='Total studies')
ax2.plot(doi_year['year'], doi_year['pct_doi'], color='#E84393',
         linewidth=2.5, marker='o', markersize=7, label='% with DOI')

ax1.set_xlabel('Year', fontsize=12)
ax1.set_ylabel('Total Studies', fontsize=12, color='#4C8BE8')
ax2.set_ylabel('% with DOI', fontsize=12, color='#E84393')
ax1.set_title('Total Studies vs DOI Coverage per Year', fontsize=16, fontweight='bold', pad=20)
ax1.xaxis.set_major_locator(mticker.MultipleLocator(1))

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc='upper left')

plt.tight_layout()
plt.savefig(f'{CHARTS_PATH}/04_doi_coverage.png', bbox_inches='tight')
plt.show()
print('Chart saved ✅')

---
## 7. Chart 5 — Top 15 collections by number of images

In [ ]:
top15 = df_c.nlargest(15, 'number_of_images')[['name', 'number_of_images', 'year']].copy()
top15['name'] = top15['name'].str[:40]

fig, ax = plt.subplots(figsize=(14, 8))

colors = sns.color_palette('viridis', len(top15))
bars = ax.barh(top15['name'], top15['number_of_images'], color=colors, edgecolor='white')

for bar in bars:
    width = bar.get_width()
    ax.text(width + 5, bar.get_y() + bar.get_height()/2,
            f'{int(width):,}', va='center', fontsize=9)

ax.set_title('Top 15 Collections by Number of Brain Maps', fontsize=16, fontweight='bold', pad=20)
ax.set_xlabel('Number of Images', fontsize=12)
ax.invert_yaxis()

plt.tight_layout()
plt.savefig(f'{CHARTS_PATH}/05_top_collections.png', bbox_inches='tight')
plt.show()
print('Chart saved ✅')

---
## 8. Summary

In [ ]:
print('=' * 50)
print('VISUALIZATION SUMMARY')
print('=' * 50)
print('Charts created and saved to Google Drive:')
print('  01_studies_per_year.png')
print('  02_growth_rate.png')
print('  03_modality_breakdown.png')
print('  04_doi_coverage.png')
print('  05_top_collections.png')
print()
print('Next step → 05_summary.ipynb 🚀')

---
## ✅ What we accomplished

- Created 5 professional charts from real neuroscience data
- Used bar charts, line plots, pie charts and dual-axis charts
- Added annotations, labels and professional styling
- Saved all charts as PNG files to Google Drive

**Next notebook:** `05_summary.ipynb` — consolidate all findings into a final narrative.